<a href="https://colab.research.google.com/github/lee-woonju/study-hongong-mldl/blob/main/05_2_%EA%B5%90%EC%B0%A8_%EA%B2%80%EC%A6%9D%EA%B3%BC_%EA%B7%B8%EB%A6%AC%EB%93%9C_%EC%84%9C%EC%B9%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교차 검증과 그리드 서치
## KEYWORD : 검증 세트, 교차 검증, 그리드 서치, 랜덤 서치
검증 세트가 필요한 이유를 이해하고 교차 검증에 대해 배워보자!



# 검증 세트
> 훈련 세트를 또 나누는 것

- 훈련세트 80% | 테스트 세트 20%
- 훈련세트 60% | **검증 세트 20%** | 테스트 세트 20%





In [56]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

# 훈련/테스트 세트로 나누기
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

print(train_input.shape, test_input.shape)

# 검증 세트
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state=42)

print(sub_input.shape, val_input.shape)


(5197, 3) (1300, 3)
(4157, 3) (1040, 3)


In [57]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


# 교차 검증
보통 많은 데이터를 훈련에 사용할수록 좋은 모델이 만들어지는데, 검증 세트를 만드느라 훈련 세트가 줄었음

> **교차 검증**은 검증 세트를 떼어 내어 평가하는 과정을 여러 번 반복한 후, 점수를 평균하여 최종 검증 점수를 얻음

## cross_validate()
cross_validate( 평가할 모델 객체, 훈련 세트 전체 )


In [58]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)


{'fit_time': array([0.01035166, 0.00866151, 0.00798559, 0.00905418, 0.00806212]), 'score_time': array([0.00199413, 0.00229383, 0.00160694, 0.00255728, 0.00223064]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


- fit_time, score_time, test_score 키를 가진 딕셔너리를 반환
- 모델을 훈련하는 시간, 검증하는 시간
  - 각 키마다 5개의 숫자(왜냐하면, 기본 5-폴드 교차 검증 수행하기 떄문에)
  - 이름은 test_score지만 검증 폴드의 점수!
- 교차 검증의 최종 점수는 test_score 키에 담긴 5개 점수의 평균



In [59]:
import numpy as np
print(np.mean(scores['test_score']))


0.855300214703487


### 교차 검증을 할때 훈련 세트를 섞으려면 분할기를 지정
- **회귀** 모델일 경우 KFold 분할기
- **분류** 모델일 경우 타깃 클래스를 골고루 나누기 위해 StratifiedKFold 사용
  - Stratified = 계층적 = 정답 비율을 원본과 똑같이 유지한다는 뜻


In [60]:
# 앞에 수행한 교차 검증은 다음 코드와 동일
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

# 훈련 세트를 섞은 후 10-폴드 교차 검증을 수행하려면 ?
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))


0.855300214703487
0.8574181117533719


# 하이퍼파라미터 튜닝
여러 개의 매개변수를 동시에 바꿔가며 최적의 값을 찾아야 해!

## GridSearchCV 클래스
for문으로 직접 구현할 수도 있지만, 사이킷런에서 제공하는 그리드 서치 클래스 사용




In [61]:
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease' : [00.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

# 그리드 서치 객체
# 결정 트리 객체를 생성하자마자 바로 그리드 서치 객체에 전달
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

# 그리드 서치 객체에 fit 호출
gs.fit(train_input, train_target)



GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

> 그리드 서치는 훈련이 끝나면 교차 검증 점수가 가장 높은 모델의 매개변수 조합으로 전체 훈련 세트에서 자동으로 다시 모델을 훈련!

$⇒$ 이 모델은 gs 객체의 **best_estimator_** 속성에 저장됨

In [62]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target ))

0.9615162593804117


$⇒$ 그리드 서치로 찾은 최적의 매개변수는 **best_params_** 속성에 저장


In [63]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


$⇒$ 각 매개변수에서 수행한 교차 검증의 평균 점수는 **cv_results_** 속성의 **mean_test_score**에 저장

In [64]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


$⇒$ gs 객체의 **best_index_** 속성을 사용해 가장 높은 값의 인덱스를 얻을 수 있음


In [65]:
print(gs.cv_results_['params'][gs.best_index_])

{'min_impurity_decrease': 0.0001}


$→$ gs.best_params_ 와 동일


> 이 과정을 정리
1. 탐색할 매개 변수 지정
2. 훈련 세트에서 그리드 서치를 수행하여 최상의 평균 점수가 나오는 매개변수 조합 찾기 $→$ 이 조합은 그리드 서치 객체에 저장
3. 그리드 서치는 최상의 매개변수에서, **전체** 훈련 세트를 사용해 최종 모델을 훈련함 $→$ 이 모델도 그리드 서치 객체에 저장됨

> 조금 더 복잡하게 해보기


In [66]:
params = {'min_impurity_decrease' : np.arange(0.0001, 0.001, 0.0001), # 0.0001에서 시작해서 0.001까지 0.0001씩 더한 배열
          'max_depth' : range(5, 20, 1),  # 5에서 20까지 1씩 증가하면서 15개의 값을 만듦
          'min_samples_split' : range(2, 100, 10) # 2에서 100까지 증가하면서 10개의 값을 만듦
          }


>이 매개변수로 수행할 교차 검증의 수 = 9 x 15 x 10 = 1,350 개<br>
>기본 5-폴드 교차 검증을 수행하므로<br>
>만들어지는 모델의 수는 = 1,350 x 5 = 6,750

In [55]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1)
gs.fit(train_input, train_target)

# 최상의 매개변수 조합 확인
print(gs.best_params_)


{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}
0.8683865773302731


In [67]:

# 최상의 교차 검증 점수 확인
print(np.max(gs.cv_results_['mean_test_score']))

0.8681929740134745


> 탐색할 매개변수의 간격을 더 좁거나 넓은 간격으로 시도해 볼 수 있지 않을까?


## 랜덤 서치
매개변수 값의 목록을 전달하는 것이 아니라 매개변수를 샘플링할 수 있는 **확률 분포 객체**를 전달
> scipy 싸이파이 라이브러리 사용


In [70]:
from scipy.stats import uniform, randint
# uniform(실수), randint(정수)
# 주어진 범위에서 고르게 = 균등 분포에서 샘플링

rgen = randint(0, 10)
rgen.rvs(10)

np.unique(rgen.rvs(1000), return_counts=True)

ugen = uniform(0, 1)
ugen.rvs(10)

array([0.59751449, 0.09486372, 0.59295715, 0.00904104, 0.54355379,
       0.63028043, 0.05821704, 0.65216978, 0.05365591, 0.60939086])

In [73]:
# 탐색할 매개변수의 딕셔너리
# min_samples_leaf : 리프 노드가 되기 위한 최소 샘플의 개수
# 어떤 노드가 분할하여 만들어질 자식 노드의 샘플 수가 이 값보다 작을 경우 분할하지 않음
# 비슷한 거 : min_impurity_decrease

params = {'min_impurity_decrease' : uniform(0.0001, 0.001),
          'max_depth' : randint(20, 50),
          'min_samples_split' : randint(2, 25),
          'min_samples_leaf' : randint(1, 25),
          }


from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

# 최적의 매개변수 조합
print(rs.best_params_)


{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [75]:
# 최고의 교차 검증 점수
print(np.max(rs.cv_results_['mean_test_score']))

dt = rs.best_estimator_ # 최적의 매개변수 조합으로 훈련된 모델
print(dt.score(test_input, test_target))


0.8695428296438884
0.86


In [77]:
# DecisionTreeClassfier splitter='random' 결과
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42, splitter='random'), params, n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

print(rs.best_params_)

dt = rs.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.786923076923077


---
# [요약] 테스트 세트는 테스트할 때만
* 훈련 세트를 나눠 **검증 세트** 만들기
* cross_validate(교차검증) : 검증 세트를 떼어 내어 평가하는 과정을 여러 번 반복
  - 훈련 세트를 섞는 기준 = 분할기 지정
  - 회귀 모델 $→$ KFold 분할기
  - 분류 모델 $→$ StratifiedKFold
* 하이퍼파라미터 튜닝
  - 그리드 서치 GridSearchCV : 교차 검증으로 하이퍼파라미터 탐색 수행
  - 랜덤 서치 RandomizedSearchCV : 교차 검증으로 **랜덤한** 하이퍼파라미터 수행
* 교차 검증 수행 후 아래정보들을 저장
  - 최적의 매개변수 조합 : .best_params_
  - 최적의 매개변수 조합으로 훈련된 모델 : .best_estimator_
* 마지막으로 테스트 세트 점수 확인
